# 06 · An MLP for regression, end to end

This is the first complete "train a network" task — the shape of homework sub-tasks D, E and F: iterators → model class → training loop with validation → curves → test loss → example predictions. Everything from notebooks 01–05 comes together.

---
## Theory in 60 seconds

- **MLP**: `Linear → activation → Linear → activation → … → Linear`. Hidden layers with a non-linearity (ReLU: `max(0, z)`) let the network represent curved relationships and interactions between features.
- **Output layer for regression**: one neuron per predicted quantity, **no activation** (the output must be free to take any real value). Loss: **MSE**.
- **Three data splits, three jobs**: *training* set → gradient updates; *validation* set → decisions made by you (number of epochs, learning rate, architecture); *test* set → one final, honest number.
- `model.train()` / `model.eval()` only set a flag (it matters for layers such as dropout); `torch.no_grad()` switches off graph recording. For evaluation use both.
- Reading curves: both losses high and flat → under-fitting (or learning rate too low); training loss falling while validation loss rises → over-fitting.

In [ ]:
import sys; sys.path.insert(0, "..")
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from dnd_utils import check, load_houses

d = load_houses()
X_train, y_train = d["X_train"], d["y_train"]          # (720, 13) and (720,)  NumPy float32; y = price in k€
X_val, y_val = d["X_val"], d["y_val"]
X_test, y_test = d["X_test"], d["y_test"]
print(X_train.shape, y_train.shape, "| mean price", y_train.mean().round(1), "std", y_train.std().round(1))

---
## Exercise 6.1 · Data loaders (repetition from 05 — from memory!)

`Dataset` subclass + three loaders, batch size 32. Batches: `(batch, 13)` and `(batch, 1)`, `float32`.

In [ ]:
loader_train = None
loader_val = None
loader_test = None

In [ ]:
check("train batch: (32, 13) and (32, 1), float32", lambda: (lambda b: b[0].shape == (32, 13) and b[1].shape == (32, 1) and b[1].dtype == torch.float32)(next(iter(loader_train))))
check("test loader keeps the order", lambda: torch.equal(torch.cat([y for _, y in loader_test])[:, 0], torch.tensor(y_test)))

---
## Exercise 6.2 · The model class, two ways

Architecture: `13 → 32 → 16 → 1`, ReLU after the two hidden layers.

**a)** Before writing code: how many trainable parameters?

In [ ]:
n_params_by_hand = None

**b)** Write `RegMLPExplicit`: three `nn.Linear` attributes created in `__init__`, and a `forward` that applies them step by step with `torch.relu` (or `nn.functional.relu`).

**c)** Write `RegMLP`: the same network with an `nn.Sequential`.

Both take `n_inputs` as constructor argument.

In [ ]:
class RegMLPExplicit(nn.Module):
    pass


class RegMLP(nn.Module):
    pass

In [ ]:
_xb = torch.randn(5, 13)
for cls_name in ("RegMLPExplicit", "RegMLP"):
    check(f"{cls_name}: output shape (5, 1)", lambda: globals()[cls_name](13)(_xb).shape == (5, 1))
    check(f"{cls_name}: 993 parameters", lambda: sum(p.numel() for p in globals()[cls_name](13).parameters()) == 993 == n_params_by_hand)
    check(f"{cls_name}: output can be negative (no activation at the end)", lambda: (lambda m: (m(torch.randn(500, 13)) < 0).any().item() and (m(torch.randn(500, 13)) > 0).any().item())((torch.manual_seed(4), globals()[cls_name](13))[1]))

What happens if you forget `super().__init__()`? And what if you create a layer inside `forward` instead of `__init__` (e.g. `return nn.Linear(16, 1)(h)`)?

✍️ **Your answer** (double-click, write it in your own words *before* opening the solution):

...

---
## Exercise 6.3 · Why the non-linearity is essential

Claim: without an activation in between, two linear layers are exactly **one** linear layer.

Given `lin1` (13 → 8) and `lin2` (8 → 1), compute the weight matrix `W_eq` (shape `(1, 13)`) and bias `b_eq` (shape `(1,)`) of the single equivalent layer. Derive it on paper: substitute `h = W₁x + b₁` into `W₂h + b₂`.

In [ ]:
torch.manual_seed(0)
lin1, lin2 = nn.Linear(13, 8), nn.Linear(8, 1)
xb = torch.randn(6, 13)

W_eq = None
b_eq = None

In [ ]:
check("shapes", lambda: W_eq.shape == (1, 13) and b_eq.shape == (1,))
check("lin2(lin1(x)) == x @ W_eq.T + b_eq", lambda: torch.allclose(lin2(lin1(xb)), xb @ W_eq.T + b_eq, atol=1e-5))
check("with a ReLU in between it is NOT the same function", lambda: not torch.allclose(lin2(torch.relu(lin1(xb))), xb @ W_eq.T + b_eq, atol=1e-3))

So a "deep" network without activations has exactly the expressive power of linear regression, however many layers it has. The ReLU between the layers is what buys the extra power.

---
## Exercise 6.4 · One epoch of training, one pass of evaluation

Write the two building blocks:

- `train_one_epoch(model, loader, loss_fn, optimizer)` — puts the model in training mode, performs one update per batch, returns the **mean of the batch losses** (a Python float).
- `evaluate(model, loader, loss_fn)` — evaluation mode, no gradients, no updates; returns the mean of the batch losses.

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer):
    raise NotImplementedError

def evaluate(model, loader, loss_fn):
    raise NotImplementedError

In [ ]:
torch.manual_seed(0)
_m = None
try: _m = RegMLP(13)
except Exception: pass
_before = [p.clone() for p in _m.parameters()] if _m is not None else None
check("evaluate returns a float and does not change the weights", lambda: isinstance(evaluate(_m, loader_val, nn.MSELoss()), float) and all(torch.equal(a, b) for a, b in zip(_before, _m.parameters())))
check("evaluate leaves no gradients behind", lambda: all(p.grad is None for p in _m.parameters()))
check("train_one_epoch returns a float and changes the weights", lambda: isinstance(train_one_epoch(_m, loader_train, nn.MSELoss(), torch.optim.Adam(_m.parameters(), lr=0.01)), float) and not all(torch.equal(a, b) for a, b in zip(_before, _m.parameters())))
check("model is in the right mode afterwards", lambda: (evaluate(_m, loader_val, nn.MSELoss()), _m.training)[1] is False)

---
## Exercise 6.5 · The training run

Train a **fresh** `RegMLP` for 80 epochs with `nn.MSELoss` and `Adam(lr=0.01)`. After every epoch store the training loss and the validation loss in `train_losses` / `val_losses`.

Start the cell with `torch.manual_seed(0)` **and create the model inside this cell** (see the question below).

In [ ]:
torch.manual_seed(0)
model = None
train_losses, val_losses = [], []

In [ ]:
check("80 values in each list", lambda: len(train_losses) == len(val_losses) == 80)
check("the first epoch starts very high (tens of thousands)", lambda: train_losses[0] > 10000)
check("training loss ends below 400", lambda: train_losses[-1] < 400)
check("validation loss ends below 450", lambda: val_losses[-1] < 450)

Why must the model (and the optimizer) be created **inside** the training cell? What would you see if you ran a training cell twice with the model created in an earlier cell?

✍️ **Your answer** (double-click, write it in your own words *before* opening the solution):

...

---
## Exercise 6.6 · Plot the curves

One figure, both curves, axis labels, **a legend**. The first epochs are so high that the interesting part is squashed — set the y-limits so that the difference between the two curves near the end is clearly visible.

In [ ]:
# your plot

Why is the loss of the very first epoch around 40 000? (Hint: what does an untrained network output, and what is the mean price?)

✍️ **Your answer** (double-click, write it in your own words *before* opening the solution):

...

---
## Exercise 6.7 · The test set: one honest number, and context for it

- `test_mse`: the MSE on the **full** test set. Collect the predictions and labels of all batches, concatenate, compute the loss once.
- `baseline_mse`: the MSE on the test set of always predicting the **training** mean price.
- Also keep `preds` and `trues`, tensors of shape `(240, 1)`.

In [ ]:
preds = None
trues = None
test_mse = None
baseline_mse = None

In [ ]:
check("preds and trues have shape (240, 1)", lambda: preds.shape == trues.shape == (240, 1))
check("trues are the test labels in order", lambda: torch.equal(trues[:, 0], torch.tensor(y_test)))
check("baseline_mse", lambda: abs(baseline_mse - float(((y_test - y_train.mean()) ** 2).mean())) < 1)
check("test_mse is consistent with preds", lambda: abs(test_mse - ((preds - trues) ** 2).mean().item()) < 1e-2)
check("the network beats the baseline by a wide margin (test MSE < 300)", lambda: test_mse < 300)

For reference: a plain **linear regression** on the same 13 inputs reaches a test MSE of about 335, and the noise built into the data has variance 144 — no model can do better than that. Where does your network sit between the two?

Now print the predicted and the true price of the first 8 test houses, one per line.

In [ ]:
# your code

---
## Exercise 6.8 · Experiment: the learning rate

Write `run(lr, n_epochs=80, seed=0)` that trains a fresh `RegMLP` with `Adam(lr=lr)` and returns `(train_losses, val_losses)`. Then run it for `lr = 0.0001`, `0.001`, `0.01` and `0.1`, and plot the four validation curves in one figure (log y-axis: `plt.yscale("log")`).

In [ ]:
def run(lr, n_epochs=80, seed=0):
    raise NotImplementedError

results = {}        # lr -> (train_losses, val_losses)

In [ ]:
check("four runs", lambda: sorted(results) == [0.0001, 0.001, 0.01, 0.1])
check("lr=0.0001 is still far from trained after 80 epochs", lambda: results[0.0001][1][-1] > 5 * results[0.01][1][-1])
check("lr=0.01 beats lr=0.001 after 80 epochs", lambda: results[0.01][1][-1] < results[0.001][1][-1])

With `lr = 0.0001` both final losses are still enormous. A classmate says "the model is under-fitting, we need a bigger network". What is the better diagnosis, and which feature of the curve tells you?

✍️ **Your answer** (double-click, write it in your own words *before* opening the solution):

...

---
## Exercise 6.9 · Reproduce the famous bug

In notebook 01 you predicted what happens when the labels have shape `(batch,)` while the model outputs `(batch, 1)`. Now watch it happen. The dataset below returns **scalar** labels. Train a fresh `RegMLP` on `loader_bug` for 40 epochs (`Adam`, `lr=0.01`) with your `train_one_epoch`, then compute:

- `bug_final_loss`: the last training loss,
- `bug_pred_std`: the standard deviation of the model's predictions on the test inputs.

PyTorch prints a `UserWarning` about the sizes — in the test, **read warnings**; this one is the only signal you get.

In [ ]:
class BuggyDataset(Dataset):
    def __init__(self, x, y):
        self.xs, self.ys = torch.tensor(x), torch.tensor(y)
    def __len__(self):
        return self.xs.shape[0]
    def __getitem__(self, idx):
        return self.xs[idx], self.ys[idx]             # <- scalar label: batches have shape (batch,)

loader_bug = DataLoader(BuggyDataset(X_train, y_train), batch_size=32, shuffle=True)

In [ ]:
bug_final_loss = None
bug_pred_std = None

In [ ]:
check("the loss is stuck near the variance of the prices", lambda: 0.7 * y_train.var() < bug_final_loss < 1.3 * y_train.var())
check("the predictions are almost constant", lambda: bug_pred_std < 0.2 * y_test.std())

Commit the fingerprint to memory: **loss stuck at about the variance of the target + nearly identical predictions for all inputs = label shape bug** (or labels shuffled independently of inputs — same fingerprint).

---
## Blank-page drill (the big one)

Open a **new, empty notebook**. Without looking at anything, and with the clock running (target: under 25 minutes), write the complete regression task:

1. `from dnd_utils import load_houses` and get the six arrays (this is the only thing you may copy);
2. `Dataset` class and three `DataLoader`s;
3. an MLP class `13 → 25 → 25 → 1` with ReLU;
4. a training loop with training **and** validation loss per epoch (write the loop inline this time, not with helper functions);
5. the loss plot with legend;
6. the test MSE on the full test set, and 5 example predictions next to the true values.

Then compare with the solution of this notebook and note down **every line you had to think about** — those are your personal weak spots for the next repetition.

In [ ]:
# (do this one in a new notebook)